In [ ]:
!pip install ragas[all]
!pip install litellm

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.1 MB/s eta 0:00:00
   

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 136.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 138.8 MB/s eta 0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.76.0
    Uninstalling grpcio-1.76.0:
      Successfully uninstalled grpcio-1.76.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.21.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 0.46.2 which is incompatible.
grpcio-status 1.71.2 requires grpcio>=1.71.2, but you have grpcio 1.67.1 which is incompatible.


In [ ]:
# Import libraries and setup config
import os
import pandas as pd
from google.colab import userdata
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecision, ContextRecall, Faithfulness, AnswerCorrectness, FactualCorrectness
from ragas.embeddings.base import embedding_factory

# Set the OPENAI_API_KEY environment variable from Colab secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Setup LLM
client = AsyncOpenAI()
# Increase max_tokens for the LLM to prevent truncation of structured outputs
llm = llm_factory("gpt-4o-mini", client=client, max_tokens=4096)
embeddings = embedding_factory("openai", model="text-embedding-3-small", client=client)

/tmp/ipython-input-2170802873.py:11: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextPrecision, ContextRecall, Faithfulness, AnswerCorrectness
/tmp/ipython-input-2170802873.py:11: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import ContextPrecision, ContextRecall, Faithfulness, AnswerCorrectness
/tmp/ipython-input-2170802873.py:11: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import ContextPrecision, Co

In [ ]:
datasets = pd.read_excel('/content/drive/MyDrive/RAG/datasets/datasets.xlsx', sheet_name="final")
answers = pd.read_csv('/content/drive/MyDrive/RAG/datasets/adv_rag_with_gemma_result_20260101.csv')

In [ ]:
# Create qa_data by combining datasets and answers_df
# Ensure alignment by index as they correspond to the same dataset
qa_data = pd.DataFrame()

# Map columns as requested
qa_data['user_input'] = datasets['question']
qa_data['answer_ref'] = datasets['answer_ref']
qa_data['answer'] = answers_df['answer']
qa_data['reference'] = datasets['passage_1_text']
qa_data['retrieved_contexts_1'] = answers_df['page_content_1']
qa_data['retrieved_contexts_2'] = answers_df['page_content_2']
qa_data['retrieved_contexts_3'] = answers_df['page_content_3']

# Create 'retrieved_contexts' list column for Ragas evaluation
# Filter out None or NaN values
def create_context_list(row):
    contexts = [row['retrieved_contexts_1'], row['retrieved_contexts_2'], row['retrieved_contexts_3']]
    return [c for c in contexts if pd.notna(c) and c is not None]

qa_data['retrieved_contexts'] = qa_data.apply(create_context_list, axis=1)

# Reset index to get the 'index' column
qa_data = qa_data.reset_index()

# Reorder columns to match request (plus retrieved_contexts for compatibility)
desired_columns = ['index', 'user_input', 'answer_ref', 'answer', 'reference',
                   'retrieved_contexts_1', 'retrieved_contexts_2', 'retrieved_contexts_3', 'retrieved_contexts']
qa_data = qa_data[desired_columns]

display(qa_data.head())

,index,user_input,answer_ref,answer,reference,retrieved_contexts_1,retrieved_contexts_2,retrieved_contexts_3,retrieved_contexts
0,0,Which month is set aside each year in the UAE ...,March,The UAE designates March annually as the month...,Building on the Abu Dhabi Reads campaign launc...,international reading performance benchmark fo...,international reading performance benchmark fo...,international reading performance benchmark fo...,[international reading performance benchmark f...
1,1,"How many public, private, and charter schools ...","Abu Dhabi has 218 public schools, 203 private ...","Based on the provided information, there are a...",There are 218 public schools with a student po...,There are 218 public schools with a student po...,There are 218 public schools with a student po...,There are 218 public schools with a student po...,[There are 218 public schools with a student p...
2,2,At which year levels are Australian students a...,Australian students are assessed in NAPLAN in ...,NaN,NAPLAN provides data on the achievement of all...,None,None,None,[]
3,3,What grading scale must Australian schools use...,Australian schools must use a five-point A–E g...,Australian schools utilize a five-point gradin...,All schools are required to report student ach...,AUSTRALIA \nPIRLS 2021 ENCYCLOPEDIA 9 \n \n \n...,AUSTRALIA \nPIRLS 2021 ENCYCLOPEDIA 9 \n \n \n...,None,[AUSTRALIA \nPIRLS 2021 ENCYCLOPEDIA 9 \n \n \...
4,4,What mandatory early education requirement mus...,Austrian children must complete one compulsory...,Austrian children who fall before March 1st of...,"Since 2010, one year of Kindergarten has been ...",of children whose sixth birthday falls before ...,of children whose sixth birthday falls before ...,of children whose sixth birthday falls before ...,[of children whose sixth birthday falls before...


## Evaluation

### Context Precision and Recall

In [ ]:
%%time
import asyncio
import nest_asyncio

# Re-initialize metrics with the compatible classes imported in the setup cell
context_precision_scorer = ContextPrecision(llm=llm)
context_recall_scorer = ContextRecall(llm=llm)

context_precision_scores = []
context_recall_scores = []

async def calculate_scores():
    for index, row in qa_data.iterrows():
        print("Evaluating index: ", index)

        # Context Precision
        try:
            context_precision_result = await context_precision_scorer.ascore(
                user_input=row['user_input'],
                reference=row['reference'],
                retrieved_contexts=row['retrieved_contexts'], # Use the list column
            )
            print(f"Precision: {context_precision_result.value}")
            context_precision_scores.append(context_precision_result.value)
        except Exception as e:
            print(f"Error calculating Precision for index {index}: {e}")
            context_precision_scores.append(None)

        # Context Recall
        try:
            context_recall_result = await context_recall_scorer.ascore(
                user_input=row['user_input'],
                reference=row['reference'],
                retrieved_contexts=row['retrieved_contexts'], # Use the list column
            )
            print(f"Recall: {context_recall_result.value}")
            context_recall_scores.append(context_recall_result.value)
        except Exception as e:
            print(f"Error calculating Recall for index {index}: {e}")
            context_recall_scores.append(None)

nest_asyncio.apply()
asyncio.run(calculate_scores())

qa_data['context_precision_score'] = context_precision_scores
qa_data['context_recall_score'] = context_recall_scores

Evaluating index:  0
Error calculating Precision for index 0: 'ContextPrecision' object has no attribute 'ascore'
Error calculating Recall for index 0: 'ContextRecall' object has no attribute 'ascore'
Evaluating index:  1
Error calculating Precision for index 1: 'ContextPrecision' object has no attribute 'ascore'
Error calculating Recall for index 1: 'ContextRecall' object has no attribute 'ascore'
Evaluating index:  2
Error calculating Precision for index 2: 'ContextPrecision' object has no attribute 'ascore'
Error calculating Recall for index 2: 'ContextRecall' object has no attribute 'ascore'
Evaluating index:  3
Error calculating Precision for index 3: 'ContextPrecision' object has no attribute 'ascore'
Error calculating Recall for index 3: 'ContextRecall' object has no attribute 'ascore'
Evaluating index:  4
Error calculating Precision for index 4: 'ContextPrecision' object has no attribute 'ascore'
Error calculating Recall for index 4: 'ContextRecall' object has no attribute 'asc

In [ ]:
# Save result to CSV
qa_data.to_csv('/content/drive/MyDrive/RAG/results/eval_adv_precision_recall_gemma_20260105.csv', index=False)

### Faithfulness

In [ ]:
%%time
import asyncio
import nest_asyncio
from pydantic import ValidationError

faithfulness_scorer = Faithfulness(llm=llm)
faithfulness_scores = []

async def calculate_faithfulness():
    for index, row in qa_data.iterrows():
        print("Evaluating index: ", index)
        try:
            faithfulness_result = await faithfulness_scorer.ascore(
                user_input=row['user_input'],
                response=row['answer'],
                retrieved_contexts=row['retrieved_contexts'],
            )
            print(faithfulness_result.value)
            faithfulness_scores.append(faithfulness_result.value)
        except (ValueError, ValidationError) as e:
            print(f"Skipping evaluation for index {index} due to: {type(e).__name__} - {e}")
            faithfulness_scores.append(None)
        except Exception as e:
            print(f"An unexpected error occurred at index {index}: {type(e).__name__} - {e}")
            faithfulness_scores.append(None)

nest_asyncio.apply()
asyncio.run(calculate_faithfulness())

qa_data['faithfulness_score'] = faithfulness_scores

Evaluating index:  0
0.4
Evaluating index:  1
0.6666666666666666
Evaluating index:  2
Skipping evaluation for index 2 due to: ValueError - retrieved_contexts is missing. Please add retrieved_contexts to the test sample.
Evaluating index:  3
0.4444444444444444
Evaluating index:  4
0.8
Evaluating index:  5
0.5
Evaluating index:  6
0.9230769230769231
Evaluating index:  7
Skipping evaluation for index 7 due to: ValueError - retrieved_contexts is missing. Please add retrieved_contexts to the test sample.
Evaluating index:  8
0.4
Evaluating index:  9
0.8888888888888888
Evaluating index:  10
0.7
Evaluating index:  11
0.875
Evaluating index:  12
Skipping evaluation for index 12 due to: ValueError - retrieved_contexts is missing. Please add retrieved_contexts to the test sample.
Evaluating index:  13
0.9166666666666666
Evaluating index:  14
0.8571428571428571
Evaluating index:  15
0.07142857142857142
Evaluating index:  16
0.5555555555555556
Evaluating index:  17
Skipping evaluation for index 17

In [ ]:
qa_data.columns

Index(['index', 'user_input', 'answer_ref', 'answer', 'reference',
       'retrieved_contexts_1', 'retrieved_contexts_2', 'retrieved_contexts_3',
       'retrieved_contexts', 'faithfulness_score'],
      dtype='object')

In [ ]:
# Save result to CSV
qa_data.to_csv('/content/drive/MyDrive/RAG/results/eval_with_adv_faithfulness_gemma_20260102.csv.csv', index=False)


### Answer Correctness

In [ ]:
%%time
import asyncio
import nest_asyncio

answer_correctness_scorer = AnswerCorrectness(llm=llm, embeddings=embeddings)
answer_correctness_scores = []

async def calculate_answer_correctness():

    for index, row in qa_data.iterrows():
        print("Evaluating index: ", index)
        answer_correctness_result = await answer_correctness_scorer.ascore(
            user_input=row['user_input'],
            response=row['answer'],
            reference=row['reference'],
        )

        print(answer_correctness_result.value)

        answer_correctness_scores.append(answer_correctness_result.value)

nest_asyncio.apply()
asyncio.run(calculate_answer_correctness())

qa_data['answer_correctness_score'] = answer_correctness_scores

Evaluating index:  0


InstructorRetryException: <failed_attempts>

<generation number="1">
<exception>
    Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
</exception>
<completion>
    None
</completion>
</generation>

<generation number="2">
<exception>
    Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
</exception>
<completion>
    None
</completion>
</generation>

<generation number="3">
<exception>
    Error code: 429 - {'error': {'message': 'You exceeded your current quota, please chec

In [ ]:
# Save result to CSV
qa_data.to_csv('/content/drive/MyDrive/RAG/results/eval_adv_with_answer_correctness_gemma_20260105.csv', index=False)

### Factual Correctness

In [ ]:
%%time
import asyncio
import nest_asyncio

factual_correctness_scorer = FactualCorrectness(llm=llm)
factual_correctness_scores = []

async def calculate_factual_correctness():
    for index, row in qa_data.iterrows():
        factual_correctness_result = await factual_correctness_scorer.ascore(
            response=row['answer'],
            reference=row['reference'],
            )

        factual_correctness_scores.append(factual_correctness_result.value)

nest_asyncio.apply()
asyncio.run(calculate_factual_correctness())

qa_data['factual_correctness_score'] = factual_correctness_scores

ValidationError: 1 validation error for ClaimDecompositionInput
response
  Input should be a valid string [type=string_type, input_value=nan, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

In [ ]:
# Save result to CSV
qa_data.to_csv('/content/drive/MyDrive/RAG/results/eval_adv_with_factual_correctness_gemma_20260105.csv', index=False)